# Validation C — the full cell

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gsilvaoelker/radcoolpv-py/blob/main/docs/site/notebooks/validation_c_pv.ipynb)

**Akerboom *et al.*, *ACS Photonics* 9 (2022) 3831–3840,
[doi:10.1021/acsphotonics.2c01389](https://doi.org/10.1021/acsphotonics.2c01389)**

## The physics

Optics, heat and electricity are coupled. The absorptance sets both how much
sunlight the cell converts and how hot it runs; the temperature sets the
open-circuit voltage; the operating voltage feeds back into the balance through
luminescent emission. radcoolpv solves the operating point and the temperature
together by fixed-point iteration.

The cell is a single diode with series and shunt resistance, an Auger term, and
a radiative saturation current from detailed balance. The band gap follows
Varshni, and its wavelength cuts off every photogeneration integral:

$$E_g(T) = E_{g0} - \frac{\alpha T^2}{T+\beta}, \qquad
J_\mathrm{sc} = q\!\int_0^{\lambda_g}\!\mathrm{IQE}\,\langle A_\mathrm{Si}\rangle\,\Phi_\mathrm{sun}\,d\lambda$$

Only $A_\mathrm{Si}$ makes carriers; the *full* absorptance heats the module.
Parasitic absorption is a thermal load and nothing more.

This group needs the **lossy** silicon table, not the nonabsorbing one groups A
and B use. Lossless silicon absorbs no sunlight at all, so $J_\mathrm{sc}$
integrates to zero and the cell reports a few millivolts — a failure that looks
like a solver bug and is a materials choice.

## Main result

| Surface | $T_\mathrm{eq}$ | Efficiency | MPP | $\beta_P$ |
|---|---:|---:|---:|---:|
| Bare Au/Si | 350.5 K | 14.17% | 142.6 W/m² | −0.303 %/K |
| Flat silica | 329.3 K | 18.09% | 182.1 W/m² | −0.299 %/K |
| Silica cylinders | 327.0 K | 18.64% | 187.7 W/m² | −0.300 %/K |

The temperature drops match the paper: 21.2 K bare → flat silica against 21 K,
2.3 K flat → cylinders against 3 K, 23.5 K bare → cylinders against 24 K.

Now read the efficiency column against the absorbed sunlight: 507.5 → 598.6 →
614.3 W/m². **Two mechanisms are at work, and the numbers separate them.** The
silica cools the module, and it also acts as an antireflection coating that
lets more light in. Both raise the efficiency; the exercise below pulls them
apart.

## Set up the runtime

Colab runtimes are temporary. Run this again after a reset.

In [ ]:
import os, subprocess, sys
from pathlib import Path
from IPython.display import Markdown, display

PROJECT = Path("/content/radcoolpv-py")
if not PROJECT.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main",
                    "https://github.com/gsilvaoelker/radcoolpv-py.git", str(PROJECT)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--editable", "."],
               cwd=PROJECT, check=True)
os.chdir(PROJECT)

from radcoolpv import config, pipeline, report
print("radcoolpv ready in", PROJECT)

## The case

This is the YAML those optics came from. The upper wavelength limit is set by
gold: `RII_Olmon_2012_ev_Au` is tabulated to 24.93 µm and the loader refuses to
extrapolate rather than inventing values.

In [ ]:
%%writefile validation_c.yaml
run:
  optics: true
  thermal: true
  plots: true
  mode: standard
  write_outputs: true
  results_dir: results/validation_c

simulation:
  wavelength: {min: 0.3, max: 24.9, n: 1000}
  angles: hemispherical
  polarization: unpolarized
  hemisphere_theta_points: 8
  hemisphere_azimuth_points: 1
  s4_modes: 60

geometry:
  source: s4
  shape: cylinder
  photonic_material: sio2
  lattice: {type: hexagonal, x: 10.608811, y: 6.125}
  cylinder: {radius: 1.75, height: 2.25}

structure:
  - {material: sio2, thickness: 500.0}
  - {material: silicon, thickness: 500.0}
  - {material: gold, thickness: 0.08}
  - {material: vacuum, thickness: 0.0, terminal: true}

materials:
  sio2: PalikKitamura_SiO2
  silicon: Palik_Si            # lossy: a lossless cell makes no current
  gold: RII_Olmon_2012_ev_Au

thermal:
  ambient_temperature: 300.0
  convection_coefficient: 6.0
  voltage: {min: 0.1, max: 0.8, n: 60}
  equilibrium: auto

## The result

The optics for this structure were computed once with S4 at the converged
settings and committed, so the thermal and electrical stages run here in
seconds with no solver. Change `SURFACE` to compare the three.

In [ ]:
SURFACE = "C3_pv_cylinders"     # C1_pv_bare | C2_pv_flat_silica | C3_pv_cylinders

CASE = "validation_c.yaml"
cfg = config.load_cases(CASE)[0]
cfg.run.optics = False                                   # read, do not solve
cfg.run.optics_results = f"validation/data/{SURFACE}.txt"
report.summary(pipeline.run(cfg))

## Run it on your own data

Leave `MY_DATA = False` and this cell does nothing — the case above has already
run. Set it to `True` and it opens a file picker, wires your file into the same
case, and runs it. You never have to edit the YAML to use your own spectrum.

Your file needs wavelength in micrometres in the first column and emittance in
another; set `MY_COLUMN` to that column's index. If you are handing it a
spectrum radcoolpv itself exported, set `MY_COLUMN = None` instead — those
files already say which column is which.

**If your spectrum reaches below about 1.1 µm you also get the PV parameters.**
Above the band gap essentially everything absorbed is absorbed in the silicon,
so radcoolpv takes the silicon absorptance to equal the emittance there and
zero below; `run.json` records that this was assumed rather than solved.

In [ ]:
MY_DATA = False      # True -> pick a file and run this case on it
MY_COLUMN = 1         # emittance column; None if the file is a radcoolpv export

if MY_DATA:
    from google.colab import files
    name = next(iter(files.upload()))      # Colab saves it beside the notebook
    cfg = config.load_cases(CASE)[0]
    cfg.run.optics = False
    cfg.run.optics_results = name
    cfg.run.optics_results_emittance_column = MY_COLUMN
    report.summary(pipeline.run(cfg))
else:
    print("Ran the case above. Set MY_DATA = True to run it on your own file.")

## Recompute the optics from the geometry

The run above read a spectrum this repository computed once and committed, so it
reproduces the published numbers in seconds on a machine with no solver. To
compute it yourself from the geometry in the YAML, set the switch below.

S4 is a C++ extension with no PyPI package, so it is built from source: about
ten minutes, then twenty minutes or so for one surface of solving. Leave the switch off for a normal run.

A different material goes in here too: upload a CSV whose first line is
`lambda_um,n,k` into `radcoolpv/materials/data/`, then name it (without the
`.csv`) in the `materials:` block above before switching this on.

In [ ]:
RECOMPUTE_WITH_S4 = False       # True -> build S4 and solve the YAML above

S4_COMMIT = "9569f5e555b967a4324eb1ea593d0f9f40761a61"      # the tested revision

def build_s4():
    import importlib, importlib.util
    if importlib.util.find_spec("S4") is not None:
        print("S4 is already importable."); return
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", "build-essential", "git",
                    "libboost-all-dev", "libfftw3-dev", "liblapack-dev",
                    "libopenblas-dev", "libsuitesparse-dev"], check=True)
    src = Path("/content/S4")
    if not src.exists():
        subprocess.run(["git", "clone", "https://github.com/phoebe-p/S4.git",
                        str(src)], check=True)
    subprocess.run(["git", "checkout", S4_COMMIT], cwd=src, check=True)
    subprocess.run(["make", "-j2", "S4_pyext"], cwd=src, check=True)
    importlib.invalidate_caches()

if RECOMPUTE_WITH_S4:
    build_s4()
    report.summary(pipeline.run(config.load_cases(CASE)[0]))
else:
    print("Using the committed spectrum. Set RECOMPUTE_WITH_S4 = True to solve"
          " the YAML above instead.")

**Exercise.** Run `SURFACE = "C1_pv_bare"` and compare
`absorbed_solar_power_W_per_m2` and `efficiency_equilibrium` with the cylinders.
The bare cell is 23.5 K hotter, and at −0.30 %/K that accounts for about 0.7
efficiency points. The gap is larger than that. Where does the rest come from?